# Studio 1 — What we built in class (answer key) ✅

**OPIM 5641 · Business Decision Modeling · Dr. Dave Wanik, UConn · Sept 9, 2026**

This is the code we wrote together in Stamford, cleaned up — cell for cell, in the order we built it.
Use it to check your own notebook, catch up if you missed a step, or as the pattern for **Assignment 1**.

> Watch the 2-minute recap video on the Studio 1 page to see these cells in action.
> The full pre-built version (with the bootstrap twist) is `Studio1_PPP_DataER.ipynb` in this folder.

## 1 · Load the data

`display.max_columns` so pandas shows all 53 columns; `low_memory=False` because mixed-type columns
are exactly the kind of dirt this dataset has.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

url = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio1/ppp_ct.csv'
df = pd.read_csv(url, low_memory=False)
print(df.shape)
df.head(3)

**Check:** `(117888, 53)` — 117,888 Connecticut PPP loans, 53 columns.
Keep the [data dictionary](https://github.com/drdave-teaching/OPIM5641-notebooks/blob/main/studio1/ppp_data_dictionary.md) open.

## 2 · Triage

In [ ]:
df.info()

## 3 · Describe the money

Mean ≈ **\$84,259** but median ≈ **\$20,833** — the average is four times the typical loan.
That gap IS the right skew (same muscle as Math Check #1).

In [ ]:
df['InitialApprovalAmount'].describe()

## 4 · The log-scale histogram (this became `fig1.png`)

A raw histogram is one giant spike — a few \$10M loans stretch the axis and hide everything.
Take the **log**… but `log(0)` is undefined and some loans show \$0, so **filter `> 0` first**.
That filter is the lesson.

In [ ]:
import numpy as np

df[df['InitialApprovalAmount'] > 0]['InitialApprovalAmount'].apply(np.log).hist()
plt.savefig('fig1.png')

## 5 · Which industries got the money?

`NAICSCode` is a 6-digit industry code — the **first two digits are the sector**
([official 2-digit sector list](https://www.census.gov/naics/)). The column is float-typed and has
missing values, so the conversion has to handle both — the `pd.notna` check keeps NaN as NaN
instead of crashing on `int(nan)`.

In [ ]:
df['2digitNAICS'] = df['NAICSCode'].apply(lambda x: str(int(x))[:2] if pd.notna(x) else np.nan)
display(df['2digitNAICS'].head())

In [ ]:
mean_loan_per_naics = df.groupby('2digitNAICS')['InitialApprovalAmount'].mean().sort_values(ascending=False)
display(mean_loan_per_naics)

**Check the top of the table:** sector **33 (Manufacturing) ≈ \$232,802**, then 32 (also Manufacturing)
≈ \$209,736, then 55 (Management of Companies) ≈ \$152,360. Bigger payrolls → bigger loans — the
program working as designed.

In [ ]:
# save the table — this and fig1.png are what we pushed to GitHub in class
mean_loan_per_naics.to_csv('mean_loan_per_naics.csv')

## 6 · The GitHub habit

What we did with these artifacts: **File → Save a copy in GitHub** → your `opim5641-work` repo —
then cloned the repo with **GitHub Desktop** and saw `fig1.png` render right on github.com.
One save, three places: Colab, GitHub, your laptop.

---

# Part 2 · Retirement Monte Carlo

## 7 · One possible future

One `for` loop, one life. Seed first, so we all see the same story.

In [ ]:
np.random.seed(5641)   # same seed, same story, every run

years = 35
contribution = 12_000

balance = 0.0
path = []
for year in range(years):
    r = np.random.normal(0.07, 0.20)   # a made-up 'typical' stock year: mean 7%, sd 20%
    balance = balance * (1 + r) + contribution
    path.append(balance)

plt.plot(path)
plt.title(f'One possible life: ending balance ${balance:,.0f}')
plt.xlabel('year'); plt.ylabel('balance ($)');

Run it twice without the seed — different life every time. One run is worthless; that's the whole
reason Monte Carlo exists.

## 8 · Ten thousand futures

Wrap the life in an outer loop. `balance` resets each realization; at the last year we bank the
ending balance into `final_savings`.

In [ ]:
np.random.seed(5641)

realizations = 10_000
years = 35
contribution = 12_000

final_savings = []
for a in range(realizations):
    balance = 0.0
    for year in range(years):
        r = np.random.normal(0.07, 0.20)
        balance = balance * (1 + r) + contribution
        if year == 34:                       # the last year — store the ending balance
            final_savings.append(balance)

print(np.shape(final_savings))

In [ ]:
plt.hist(final_savings, bins=80)
plt.axvline(np.mean(final_savings), color='red', linestyle='dashed', label=f'mean ${np.mean(final_savings)/1e6:,.2f}M')
plt.legend()
plt.title('10,000 possible endings')
plt.xlabel('ending balance ($)');

**Answer questions with probabilities, not single numbers:**

In [ ]:
final_savings = np.array(final_savings)
print(f'P(ending >= $1M)   = {(final_savings >= 1_000_000).mean():.1%}')
print(f'P(ending <  $500k) = {(final_savings < 500_000).mean():.1%}')

## Where to go next

The pre-built notebook `Studio1_Retirement_MC.ipynb` (same folder) takes this two steps further:
the **spaghetti plot** of every path, and the twist we previewed — replacing `np.random.normal`
with a **bootstrap of 98 years of real S&P 500 returns** (`sp['sp500_total_return'].sample(...)`).
Fat tails change the answer; go see by how much.

**Assignment 1 is exactly this recipe on a dataset you choose:** explore what you have, then
estimate the uncertain input from your data and simulate the future you don't have.